# Femur (FL) Dataset Inspection and Follow-Up Analysis

## 1. Purpose & Protocol
This benchmark notebook inspects and documents the Fetal Femur Length (FL) dataset across both FP (Fetal Planes) and UCL biometry cohorts:
1. **Outliers Analysis**: Identification and clinical/technical root cause classification for all images with FL > 85 mm (long outliers) and the 10 shortest FL measurements (short outliers), complete with visual endpoint overlays.
2. **Missing Rate Exclusions**: Audit of the 3 FP images with missing pixel-to-mm calibration rates (`px_to_mm_rate.isna()`), patient cross-plane checks, and verified programmatic exclusion from training and evaluation.
3. **UCL Patient 125 Rule**: Rationale and empirical effect of the `~image_name.str.startswith("125_")` exclusion rule.
4. **Hadlock 3-Plane Feasibility**: Cross-reference check of whether FP patients with Femur and Abdomen images also possess head-plane data for 3-plane Hadlock EFW estimation.
5. **Split Integrity**: Clean verification of patient-isolated Train, Validation, and Test splits guaranteeing strictly 0 patient overlap.

All figures and tables are dynamically generated and verified from local datasets and result artifacts.


In [1]:
import sys
from pathlib import Path
import subprocess

# Locate work root
work_dir = Path.cwd()
while work_dir.name != "work" and work_dir.parent != work_dir:
    work_dir = work_dir.parent
if str(work_dir / "src") not in sys.path:
    sys.path.insert(0, str(work_dir / "src"))

print(f"Work directory: {work_dir}")
script_path = work_dir / "scripts" / "21_femur_data_followup.py"
print(f"Executing {script_path.name}...")

proc = subprocess.run(
    [sys.executable, str(script_path)],
    cwd=str(work_dir),
    capture_output=True,
    text=True,
    check=True,
)
print(proc.stdout)


Work directory: C:\fetal-github-clone\work
Executing 21_femur_data_followup.py...


Saved femur follow-up results to C:\fetal-github-clone\work\docs\results\femur\femur_data_followup.json

FEMUR (FL) DATA FOLLOW-UP REPORT

--- 1. OUTLIERS (FL > 85 mm and 10 Shortest FL) ---
Filename                         Patient          Src  Split  FL (mm)  Rate     Cause                         
--------------------------------------------------------------------------------------------------------------
Patient00975_Plane5_2_of_2.png   FP_Patient00975  FP   train  138.20   0.2941   wrong endpoints               
Patient01785_Plane5_1_of_1.png   FP_Patient01785  FP   train  123.43   0.3333   wrong px_to_mm_rate           
Patient01246_Plane5_2_of_2.png   FP_Patient01246  FP   train  102.99   0.2632   both wrong px_to_mm_rate and wrong endpoints
Patient00855_Plane5_1_of_3.png   FP_Patient00855  FP   train  102.67   0.2500   both wrong px_to_mm_rate and wrong endpoints
Patient00855_Plane5_2_of_3.png   FP_Patient00855  FP   train  101.99   0.2500   both wrong px_to_mm_rate and wrong 

In [2]:
import json
from pathlib import Path
import pandas as pd
from IPython.display import display, Markdown

results_file = work_dir / "docs" / "results" / "femur" / "femur_data_followup.json"
assert results_file.is_file(), f"Missing results at {results_file}"

with open(results_file, "r", encoding="utf-8") as f:
    data = json.load(f)

print("=== 1. OUTLIERS (FL > 85 mm and 10 Shortest FL) ===")
outliers_df = pd.DataFrame(data["outliers"])
display(outliers_df[["category", "filename", "patient_id", "dataset", "split", "fl_mm", "px_to_mm_rate", "cause"]])

print("\n=== 2. MISSING CALIBRATION RATE (FP) ===")
missing = data["missing_rate_audit"]
print(f"Excluded images: {missing['missing_rate_images']}")
print(f"Patients have other femur data: {missing['same_patient_other_femur_images_with_rate']}")
print(f"Patients have other AC data: {missing['same_patient_ac_images']}")
print(f"Exclusion rule: {missing['code_exclusion_rule']}")

print("\n=== 3. UCL EXCLUDES 125 RULE ===")
ucl_rule = data["ucl_excludes_125_rule"]
print(f"Rule: {ucl_rule['rule']}")
print(f"Raw UCL 125 rows: {ucl_rule['ucl_raw_annotations_count_matching_125']}")
print(f"Calibrated UCL 125 rows: {ucl_rule['ucl_calibrated_px_to_mm_count_matching_125']}")
print(f"Images removed from calibrated pipeline: {ucl_rule['images_removed_from_calibrated_pipeline']}")

print("\n=== 4. HADLOCK 3-PLANE PATIENT CHECK ===")
hadlock = data["hadlock_3plane_check"]
print(f"FP Femur patients: {hadlock['fp_patients_with_femur']}")
print(f"FP AC patients: {hadlock['fp_patients_with_ac']}")
print(f"FP patients with Femur + AC: {hadlock['fp_patients_with_femur_and_ac']}")
print(f"FP has head annotations: {hadlock['fp_has_head_plane_annotations']}")
print(f"FP has head images: {hadlock['fp_has_head_plane_images']}")
print(f"Patients with Head + Abdomen + Femur: {hadlock['count_patients_with_head_abdomen_femur_together']}")

print("\n=== 5. SPLIT COUNTS AFTER EXCLUSIONS ===")
splits = data["split_counts_after_exclusions"]
split_rows = []
for sname, info in splits["splits"].items():
    split_rows.append({
        "Split": sname.capitalize(),
        "Images": info["images"],
        "FP Images": info["fp_images"],
        "UCL Images": info["ucl_images"],
        "Patients": info["patients"],
    })
display(pd.DataFrame(split_rows))
print(f"Total images: {splits['total_images_after_exclusions']}, Total patients: {splits['total_patients_after_exclusions']}")
p_chk = splits["patient_overlap_check"]
print(f"Zero patient overlap confirmed: {p_chk['zero_patient_overlap_confirmed']}")


=== 1. OUTLIERS (FL > 85 mm and 10 Shortest FL) ===


,category,filename,patient_id,dataset,split,fl_mm,px_to_mm_rate,cause
0,long (> 85 mm),Patient00975_Plane5_2_of_2.png,FP_Patient00975,FP,train,138.20,0.2941,wrong endpoints
1,long (> 85 mm),Patient01785_Plane5_1_of_1.png,FP_Patient01785,FP,train,123.43,0.3333,wrong px_to_mm_rate
2,long (> 85 mm),Patient01246_Plane5_2_of_2.png,FP_Patient01246,FP,train,102.99,0.2632,both wrong px_to_mm_rate and wrong endpoints
3,long (> 85 mm),Patient00855_Plane5_1_of_3.png,FP_Patient00855,FP,train,102.67,0.2500,both wrong px_to_mm_rate and wrong endpoints
4,long (> 85 mm),Patient00855_Plane5_2_of_3.png,FP_Patient00855,FP,train,101.99,0.2500,both wrong px_to_mm_rate and wrong endpoints
5,shortest 10,Patient00954_Plane5_2_of_3.png,FP_Patient00954,FP,train,13.07,0.1000,wrong endpoints
6,shortest 10,Patient01347_Plane5_2_of_2.png,FP_Patient01347,FP,train,13.63,0.0893,valid anatomical measurement
7,shortest 10,Patient00954_Plane5_3_of_3.png,FP_Patient00954,FP,train,13.87,0.1000,wrong endpoints
8,shortest 10,Patient01606_Plane5_2_of_2.png,FP_Patient01606,FP,train,14.00,0.1250,wrong endpoints
9,shortest 10,Patient01080_Plane5_1_of_2.png,FP_Patient01080,FP,train,14.66,0.1136,wrong endpoints



=== 2. MISSING CALIBRATION RATE (FP) ===
Excluded images: ['Patient00888_Plane5_1_of_1.png', 'Patient01319_Plane5_1_of_1.png', 'Patient01423_Plane5_1_of_1.png']
Patients have other femur data: 0
Patients have other AC data: 0
Exclusion rule: build_femur_data(drop_missing_rate=True) filters df[px_to_mm_rate.notna()] by default

=== 3. UCL EXCLUDES 125 RULE ===
Rule: df = df[~df['image_name'].str.startswith('125_')]
Raw UCL 125 rows: 1
Calibrated UCL 125 rows: 0
Images removed from calibrated pipeline: 0

=== 4. HADLOCK 3-PLANE PATIENT CHECK ===
FP Femur patients: 629
FP AC patients: 586
FP patients with Femur + AC: 407
FP has head annotations: False
FP has head images: False
Patients with Head + Abdomen + Femur: 0

=== 5. SPLIT COUNTS AFTER EXCLUSIONS ===


,Split,Images,FP Images,UCL Images,Patients
0,Train,451,382,69,345
1,Validation,65,52,13,47
2,Test,348,323,25,266


Total images: 864, Total patients: 658
Zero patient overlap confirmed: True


In [3]:
# Automated verification assertions
assert splits["total_images_before_exclusions"] == 867
assert splits["excluded_uncalibrated_images"] == 3
assert splits["total_images_after_exclusions"] == 864
assert splits["total_patients_after_exclusions"] == 658
assert splits["splits"]["train"]["images"] == 451
assert splits["splits"]["validation"]["images"] == 65
assert splits["splits"]["test"]["images"] == 348
assert p_chk["zero_patient_overlap_confirmed"] is True
assert hadlock["count_patients_with_head_abdomen_femur_together"] == 0
assert len(data["outliers"]) == 15
print("All benchmark assertions passed cleanly!")


All benchmark assertions passed cleanly!
